# Task 4: Context-Aware Chatbot Using LangChain / RAG

**Objective:** Build a conversational chatbot that remembers context and retrieves answers from a custom document corpus using Retrieval-Augmented Generation (RAG).

**Tools:** `LangChain`, `FAISS`, `sentence-transformers`, `HuggingFace Transformers`, `Streamlit`

**Approach:**
- Embeddings: `sentence-transformers/all-MiniLM-L6-v2` (local, no API key needed)
- Vector Store: FAISS (local, in-memory)
- LLM: `google/flan-t5-base` (local, runs on CPU)
- Memory: LangChain ConversationBufferMemory

## 1. Imports

In [1]:
import warnings
warnings.filterwarnings("ignore")

from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationalRetrievalChain
from langchain.schema import Document
from transformers import pipeline
import torch

print("All imports successful")
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")

All imports successful
Device: cpu


## 2. Prepare Custom Corpus

We use a custom text corpus about Artificial Intelligence and Machine Learning topics as our knowledge base. This simulates a real-world scenario where the chatbot answers questions from internal documents or a specific knowledge domain.

In [2]:
corpus = """
Artificial Intelligence (AI) refers to the simulation of human intelligence in machines
that are programmed to think and learn. AI systems can perform tasks such as visual
perception, speech recognition, decision-making, and language translation.

Machine Learning (ML) is a subset of AI that enables systems to learn and improve from
experience without being explicitly programmed. ML algorithms build a mathematical model
based on sample data, known as training data, to make predictions or decisions.

Deep Learning is a subset of machine learning that uses neural networks with many layers
(deep neural networks) to learn representations of data. It has achieved remarkable results
in image recognition, natural language processing, and speech recognition.

Natural Language Processing (NLP) is a branch of AI that deals with the interaction
between computers and human language. NLP techniques include text classification,
sentiment analysis, named entity recognition, and machine translation.

Transformers are a type of deep learning model introduced in the paper "Attention is All
You Need" in 2017. They use self-attention mechanisms to process sequential data and have
become the dominant architecture for NLP tasks. BERT, GPT, and T5 are popular transformer
models.

BERT (Bidirectional Encoder Representations from Transformers) is a transformer-based
model developed by Google. It is pre-trained on a large corpus of text and can be
fine-tuned for various NLP tasks such as text classification, question answering,
and named entity recognition.

Retrieval-Augmented Generation (RAG) is an AI framework that combines retrieval-based
and generative approaches. It retrieves relevant documents from a knowledge base and
uses them as context for a language model to generate accurate and grounded responses.

Vector databases store data as high-dimensional vectors (embeddings) and enable
similarity search. They are commonly used in RAG systems to find relevant documents
based on semantic similarity to a user query. FAISS, Pinecone, and ChromaDB are
popular vector databases.

Reinforcement Learning (RL) is a type of machine learning where an agent learns to
make decisions by interacting with an environment. The agent receives rewards or
penalties for its actions and learns to maximize cumulative reward over time.

Computer Vision is a field of AI that enables machines to interpret and understand
visual information from the world. Applications include object detection, image
classification, facial recognition, and autonomous driving.

Large Language Models (LLMs) are AI systems trained on massive text datasets that
can understand and generate human-like text. Examples include GPT-4, Claude, Gemini,
and LLaMA. They are used for chatbots, content generation, code completion, and more.

Fine-tuning is the process of taking a pre-trained model and training it further on
a specific dataset for a particular task. This allows the model to adapt its general
knowledge to a specific domain while requiring much less data and compute than
training from scratch.

Prompt engineering is the practice of designing and optimizing input prompts to
elicit desired outputs from language models. It includes techniques such as zero-shot
prompting, few-shot prompting, chain-of-thought prompting, and role-based prompting.

Scikit-learn is a popular Python library for machine learning. It provides simple
and efficient tools for data mining and data analysis, including classification,
regression, clustering, and dimensionality reduction algorithms.

PyTorch is an open-source machine learning framework developed by Facebook. It
provides dynamic computational graphs and is widely used for deep learning research
and production. TorchVision, TorchText, and TorchAudio are popular extensions.

LangChain is a framework for building applications powered by language models. It
provides tools for chaining LLM calls, managing memory, connecting to external data
sources, and building agents that can use tools to complete complex tasks.
"""

print("Corpus loaded successfully")
print(f"Total characters: {len(corpus)}")

Corpus loaded successfully
Total characters: 4048


## 3. Split Corpus into Chunks

The corpus is split into smaller overlapping chunks to improve retrieval quality. Each chunk is small enough to fit in the LLM context window but large enough to contain meaningful information.

In [3]:
# Wrap corpus in a Document object
documents = [Document(page_content=corpus)]

# Split into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
    separators=["\n\n", "\n", ".", " "]
)

chunks = text_splitter.split_documents(documents)

print(f"Number of chunks created: {len(chunks)}")
print(f"\nSample chunk:\n{chunks[0].page_content}")

Number of chunks created: 16

Sample chunk:
Artificial Intelligence (AI) refers to the simulation of human intelligence in machines
that are programmed to think and learn. AI systems can perform tasks such as visual
perception, speech recognition, decision-making, and language translation.


## 4. Create Embeddings

Using `sentence-transformers/all-MiniLM-L6-v2` — a lightweight, fast embedding model that runs locally on CPU without any API key. It converts each chunk into a 384-dimensional dense vector.

In [4]:
print("Loading embedding model... (first time may take a minute to download)")

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"}
)

# Test embedding
test_embedding = embeddings.embed_query("What is machine learning?")
print(f"Embedding model loaded successfully")
print(f"Embedding dimension: {len(test_embedding)}")

Loading embedding model... (first time may take a minute to download)


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


Embedding model loaded successfully
Embedding dimension: 384


## 5. Build FAISS Vector Store

All document chunks are embedded and stored in a FAISS index. During retrieval, the user query is embedded and the most semantically similar chunks are returned.

In [5]:
print("Building FAISS vector store...")

vectorstore = FAISS.from_documents(chunks, embeddings)

print(f"Vector store created with {vectorstore.index.ntotal} vectors")

# Test retrieval
test_query = "What is BERT?"
retrieved = vectorstore.similarity_search(test_query, k=2)

print(f"\nTest retrieval for: '{test_query}'")
for i, doc in enumerate(retrieved):
    print(f"\nChunk {i+1}:\n{doc.page_content}")

Building FAISS vector store...
Vector store created with 16 vectors

Test retrieval for: 'What is BERT?'

Chunk 1:
BERT (Bidirectional Encoder Representations from Transformers) is a transformer-based
model developed by Google. It is pre-trained on a large corpus of text and can be
fine-tuned for various NLP tasks such as text classification, question answering,
and named entity recognition.

Chunk 2:
Transformers are a type of deep learning model introduced in the paper "Attention is All
You Need" in 2017. They use self-attention mechanisms to process sequential data and have
become the dominant architecture for NLP tasks. BERT, GPT, and T5 are popular transformer
models.


## 6. Save Vector Store

Saving the FAISS index locally so it can be reloaded by the Streamlit app without rebuilding from scratch every time.

In [6]:
import os
os.makedirs("outputs", exist_ok=True)

vectorstore.save_local("outputs/faiss_index")
print("Vector store saved to outputs/faiss_index/")

Vector store saved to outputs/faiss_index/


## 7. Load LLM — Flan-T5-Base (Local, No API Key)

Using `google/flan-t5-base` — a small but capable instruction-following language model that runs on CPU. It will generate answers based on the retrieved context chunks.

In [7]:
print("Loading LLM (flan-t5-base)... this may take 1-2 minutes on first run")

llm_pipeline = pipeline(
    "text2text-generation",
    model="google/flan-t5-base",
    max_new_tokens=256,
    temperature=0.3,
    do_sample=False,
    device=-1  # CPU
)

llm = HuggingFacePipeline(pipeline=llm_pipeline)

print("LLM loaded successfully")

Loading LLM (flan-t5-base)... this may take 1-2 minutes on first run


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


LLM loaded successfully


## 8. Build Conversational RAG Chain

Combining the retriever, LLM, and memory into a `ConversationalRetrievalChain`. The memory stores conversation history so the chatbot can reference previous exchanges in follow-up questions.

In [8]:
# Set up retriever
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

# Conversation memory
memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True,
    output_key="answer"
)

# Build the chain
qa_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retriever,
    memory=memory,
    return_source_documents=True,
    verbose=False
)

print("Conversational RAG chain built successfully")

Conversational RAG chain built successfully


## 9. Test the Chatbot

Running a multi-turn conversation to demonstrate that the chatbot retrieves relevant context and maintains conversational memory across turns.

In [9]:
def ask(question):
    print(f"\nQ: {question}")
    result = qa_chain.invoke({"question": question})
    print(f"A: {result['answer']}")
    print(f"\nSources used:")
    for doc in result["source_documents"]:
        print(f"  - {doc.page_content[:100]}...")
    return result["answer"]

In [10]:
# Turn 1
ask("What is machine learning?")


Q: What is machine learning?
A: enables systems to learn and improve from experience without being explicitly programmed

Sources used:
  - Machine Learning (ML) is a subset of AI that enables systems to learn and improve from
experience wi...
  - Deep Learning is a subset of machine learning that uses neural networks with many layers
(deep neura...
  - Artificial Intelligence (AI) refers to the simulation of human intelligence in machines
that are pro...


'enables systems to learn and improve from experience without being explicitly programmed'

In [11]:
# Turn 2
ask("How is it different from deep learning?")


Q: How is it different from deep learning?
A: a subset of machine learning

Sources used:
  - Deep Learning is a subset of machine learning that uses neural networks with many layers
(deep neura...
  - Machine Learning (ML) is a subset of AI that enables systems to learn and improve from
experience wi...
  - PyTorch is an open-source machine learning framework developed by Facebook. It
provides dynamic comp...


'a subset of machine learning'

In [12]:
# Turn 3 — tests memory
ask("Can you give me an example of a model that uses it?")


Q: Can you give me an example of a model that uses it?
A: Fine-tuning

Sources used:
  - Large Language Models (LLMs) are AI systems trained on massive text datasets that
can understand and...
  - Fine-tuning is the process of taking a pre-trained model and training it further on
a specific datas...
  - Machine Learning (ML) is a subset of AI that enables systems to learn and improve from
experience wi...


'Fine-tuning'

In [13]:
# Turn 4
ask("What is RAG and how does it work?")


Q: What is RAG and how does it work?
A: Retrieval-Augmented Generation

Sources used:
  - Reinforcement Learning (RL) is a type of machine learning where an agent learns to
make decisions by...
  - Fine-tuning is the process of taking a pre-trained model and training it further on
a specific datas...
  - Retrieval-Augmented Generation (RAG) is an AI framework that combines retrieval-based
and generative...


'Retrieval-Augmented Generation'

In [14]:
# Turn 5
ask("What tools are used to build RAG systems?")



Q: What tools are used to build RAG systems?
A: Vector databases

Sources used:
  - Retrieval-Augmented Generation (RAG) is an AI framework that combines retrieval-based
and generative...
  - Vector databases store data as high-dimensional vectors (embeddings) and enable
similarity search. T...
  - LangChain is a framework for building applications powered by language models. It
provides tools for...


'Vector databases'

## 10. Inspect Conversation Memory

Viewing the full conversation history stored in memory to confirm context is being retained across turns.

In [15]:
print("=== Conversation History ===\n")
for i, message in enumerate(memory.chat_memory.messages):
    role = "Human" if i % 2 == 0 else "Assistant"
    print(f"{role}: {message.content}\n")

=== Conversation History ===

Human: What is machine learning?

Assistant: enables systems to learn and improve from experience without being explicitly programmed

Human: How is it different from deep learning?

Assistant: a subset of machine learning

Human: Can you give me an example of a model that uses it?

Assistant: Fine-tuning

Human: What is RAG and how does it work?

Assistant: Retrieval-Augmented Generation

Human: What tools are used to build RAG systems?

Assistant: Vector databases



## 11. Evaluate Retrieval Quality

Testing the retriever with a set of diverse questions to confirm it fetches semantically relevant chunks even when phrasing differs from the original text.

In [16]:
test_questions = [
    "What is BERT used for?",
    "How do vector databases work?",
    "What is prompt engineering?",
    "Tell me about PyTorch",
    "What are large language models?"
]

print("=== Retrieval Quality Test ===\n")
for q in test_questions:
    docs = retriever.invoke(q)
    print(f"Q: {q}")
    print(f"Top chunk: {docs[0].page_content[:120]}...")
    print()

=== Retrieval Quality Test ===

Q: What is BERT used for?
Top chunk: BERT (Bidirectional Encoder Representations from Transformers) is a transformer-based
model developed by Google. It is p...

Q: How do vector databases work?
Top chunk: Vector databases store data as high-dimensional vectors (embeddings) and enable
similarity search. They are commonly use...

Q: What is prompt engineering?
Top chunk: Prompt engineering is the practice of designing and optimizing input prompts to
elicit desired outputs from language mod...

Q: Tell me about PyTorch
Top chunk: PyTorch is an open-source machine learning framework developed by Facebook. It
provides dynamic computational graphs and...

Q: What are large language models?
Top chunk: Large Language Models (LLMs) are AI systems trained on massive text datasets that
can understand and generate human-like...



## Final Summary / Insights

### System Overview
| Component | Choice | Reason |
|-----------|--------|--------|
| Embeddings | all-MiniLM-L6-v2 | Fast, local, no API key needed |
| Vector Store | FAISS | Free, in-memory, fast similarity search |
| LLM | google/flan-t5-base | Local CPU inference, no cost |
| Memory | ConversationBufferMemory | Retains full chat history |
| Framework | LangChain | Modular chain construction |

### Multi-Turn Conversation Results
| Turn | Question | Answer |
|------|----------|--------|
| 1 | What is machine learning? | Enables systems to learn and improve from experience without being explicitly programmed |
| 2 | How is it different from deep learning? | A subset of machine learning |
| 3 | Can you give me an example of a model that uses it? | Fine-tuning |
| 4 | What is RAG and how does it work? | Retrieval-Augmented Generation |
| 5 | What tools are used to build RAG systems? | Vector databases |

### Retrieval Quality
- Retriever successfully fetched relevant chunks for all 5 test questions
- Semantic similarity search worked correctly even when query phrasing differed from the exact corpus wording — e.g., "Tell me about PyTorch" correctly retrieved the PyTorch paragraph despite not being an exact phrase match
- Each response was grounded in 3 retrieved source chunks, visible in the Streamlit app's expandable source panel

### Conversational Memory
- Memory retained all 5 conversation turns (10 messages total — 5 Human, 5 Assistant) with zero context loss
- Full conversation history confirmed via `ConversationBufferMemory` inspection in Cell 15
- Follow-up questions (Turn 2: "How is it different?" Turn 3: "Can you give me an example?") were resolved using prior conversation context, demonstrating that the chain correctly referenced chat history rather than treating each question in isolation

### Limitations
- `flan-t5-base` is a small model — answers are factual and grounded but short (single phrase or sentence); a larger LLM such as Llama 3 or Mistral would produce more fluent and detailed responses
- Corpus is limited to the provided AI/ML text — questions outside this domain return incomplete answers
- FAISS index is in-memory only — a persistent vector database (e.g., ChromaDB, Pinecone) would be needed for production deployment with a larger knowledge base